<a href="https://colab.research.google.com/github/samuel-harvan/TruLearn-CxC/blob/main/nli_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q -U sentence-transformers datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.3 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()

Saving nli.json to nli.json


In [5]:
import json
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from datasets import Dataset
import torch
from google.colab import drive

def train_cross_encoder():

    drive.mount('/content/drive')

    with open('nli.json', 'r') as f:
        data = json.load(f)

    #Note: 0 = contradiction label, 1 = entailment label, 2 = neutral label (refer to nli.json)
    dataset = Dataset.from_dict({
        "sentence1": [item["student_answer"] for item in data],
        "sentence2": [item["sample_answer"] for item in data],
        "label": [int(item["label"]) for item in data],
    })

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    #optimizing for memory usage. Lowered the token limit due to memory issues
    model = CrossEncoder('cross-encoder/nli-deberta-v3-base', num_labels=3, device=device)
    model.tokenizer.model_max_length = 128

    args = CrossEncoderTrainingArguments(
        output_dir='/content/drive/MyDrive/trulearn-models',
        num_train_epochs=5,
        per_device_train_batch_size=16,
        warmup_steps=100,
    )

    trainer = CrossEncoderTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
    )

    trainer.train()
    model.save_pretrained('/content/drive/MyDrive/trulearn-models/nli-model')

if __name__ == "__main__":
    train_cross_encoder()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
